## Get the data from google drive


In [3]:
import os
import zipfile

# 1. name of the source id and how it was stored
file_id = '1_5G3Cz0WQtZeTxzsq5lrTUfEnNy78WeY'
zip_name = 'Plant_Disease_Dataset.zip'

print("⚡ from sourse drive (Source Drive) directly ZIP file is downloaded...")
# it was unzipped and uploaded directly to the server without downloading to drive or pc
!gdown {file_id} -O /content/{zip_name}

print(f"\n📦 Downloaded succsesfully. {zip_name} unzipping...")

# colab RAM (Local cloud space) is quickly unziping the file
with zipfile.ZipFile(f'/content/{zip_name}', 'r') as zip_ref:
    zip_ref.extractall('/content/')

print("✅ unzip finished! ")

⚡ from sourse drive (Source Drive) directly ZIP file is downloaded...
Downloading...
From (original): https://drive.google.com/uc?id=1_5G3Cz0WQtZeTxzsq5lrTUfEnNy78WeY
From (redirected): https://drive.google.com/uc?id=1_5G3Cz0WQtZeTxzsq5lrTUfEnNy78WeY&confirm=t&uuid=97f4fbde-b1bf-49f3-8f49-ef268af3d566
To: /content/Plant_Disease_Dataset.zip
100% 1.49G/1.49G [00:17<00:00, 86.0MB/s]

📦 Downloaded succsesfully. Plant_Disease_Dataset.zip unzipping...
✅ unzip finished! 


In [4]:
import os
#listing the folders aftr unzipping
print(os.listdir('/content/'))

['.config', 'Plant_Disease_Dataset', '__MACOSX', 'Plant_Disease_Dataset.zip', 'sample_data']


## 1.Import Libraries

In [5]:
!pip install -q scikit-image scipy

In [6]:
import os
import cv2
import numpy as np
import pandas as pd
from scipy.stats import skew

## 2.Feature Extraction Function

In [7]:
def extract_leaf(image_path):
  img=cv2.imread(image_path)
  if img is None:
    return None
  img=cv2.resize(img,(256,256))

  #Leaf Segmentation (segmentation using otsu's  Thresholding)
  gray=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
  threshold_value,mask=cv2.threshold(gray,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)

  #color space conversion (HSV and Lab)
  hsv=cv2.cvtColor(img,cv2.COLOR_BGR2HSV)
  lab=cv2.cvtColor(img,cv2.COLOR_BGR2LAB)


  leaf_pixels_hsv=hsv[mask==255]
  leaf_pixels_lab=lab[mask==255]

  if len(leaf_pixels_hsv)==0:
    return None

  #colors feature extraction
  features=[]

  #color features (Color moments:Mean , Std, skew)
  channel=[leaf_pixels_hsv[:,0],leaf_pixels_hsv[:,1],leaf_pixels_lab[:,1],leaf_pixels_lab[:,2
                                                                                         ]]
  #calculate Mean.std,skew for each wave set
  for ch in channel:
    features.append(np.mean(ch))
    features.append(np.std(ch))
    features.append(skew(ch))
  #Texxture feature using GLCM
  #GLCM features
  glcm=graycomatrix(gray,distances=[1],angles=[0,np.pi/4,np.pi/2,3*np.pi/4],levels=256,symmetric=True,normed=True)


  features.append(np.mean(graycoprops(glcm,'contrast')))
  features.append(np.mean(graycoprops(glcm,'homogeneity')))
  features.append(np.mean(graycoprops(glcm,'energy')))
  features.append(np.mean(graycoprops(glcm,'correlation')))

  return features


## 3.Feature Extraction Excution

## proecess the dataset and set csv

In [8]:
import os

base = '/content/Plant_Disease_Dataset'
print("1.files included in primary folder:")
print(os.listdir(base))

# if train and test exist the what inclueds that folder:
for f in os.listdir(base):
    if f.lower() == 'train':
        print(f"\n2. '{f}' included inside:")
        print(os.listdir(os.path.join(base, f)))

1.files included in primary folder:
['.DS_Store', 'test', 'valid', 'train']

2. 'train' included inside:
['Squash___Powdery_mildew', 'Tomato___Target_Spot', '.DS_Store', 'Strawberry___Leaf_scorch', 'Tomato___Bacterial_spot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Strawberry___healthy', 'Peach___Bacterial_spot', 'Apple___Apple_scab', 'Tomato___Late_blight', 'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___healthy', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___Black_rot', 'Potato___Early_blight', 'Grape___healthy', 'Tomato___Septoria_leaf_spot', 'Corn_(maize)___Northern_Leaf_Blight', 'Tomato___Tomato_mosaic_virus', 'Apple___Black_rot', 'Orange___Haunglongbing_(Citrus_greening)', 'Potato___Late_blight', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Grape___Esca_(Black_Measles)', 'Raspberry___healthy', 'Corn_(maize)___Common_rust_', 'Blueberry___healthy', 'Corn_(maize)_

In [ ]:
import os
import cv2
import pandas as pd

# primary path
dataset_path = '/content/Plant_Disease_Dataset'
splits = ['train', 'valid', 'test']
data = []

print("Extracting features from your local Plant_Disease_Dataset... Please wait...")

# Loop through each data split (train, valid, test)
for split in splits:
    split_path = os.path.join(dataset_path, split)

    if not os.path.exists(split_path):
        continue

    #
    current_categories = [f for f in os.listdir(split_path) if os.path.isdir(os.path.join(split_path, f)) and f != '__MACOSX']

    for category in current_categories:
        folder_path = os.path.join(split_path, category)
        label = 0 if 'healthy' in category.lower() else 1

        print(f"Processing images from: {split}/{category}...")

        # Loop through all images inside the category folder
        for img_name in os.listdir(folder_path):
            img_path = os.path.join(folder_path, img_name)

            # Filter for standard image formats and ignore hidden metadata files like .DS_Store
            if img_name.lower().endswith(('.png', '.jpg', '.jpeg')) and not img_name.startswith('.'):
                # extraction funtion
                features = extract_leaf(img_path)
                if features is not None:
                    row = features + [category, label]
                    data.append(row)

# Column structure (finall=y 'category_name' and 'label' included)
columns = [
    'h_mean', 'h_std', 'h_skew',
    's_mean', 's_std', 's_skew',
    'a_mean', 'a_std', 'a_skew',
    'b_mean', 'b_std', 'b_skew',
    'contrast', 'homogeneity', 'energy', 'correlation',
    'category_name', 'label'
]

# Create DataFrame and save it as a local CSV file
df = pd.DataFrame(data, columns=columns)
df.to_csv('leaf_features.csv', index=False)
print(f"\nSuccess! 'leaf_features.csv' created.")
print(f"Total images successfully processed across all folders: {len(df)}")

Extracting features from your local Plant_Disease_Dataset... Please wait...
Processing images from: train/Tomato___Septoria_leaf_spot...
Processing images from: train/Peach___healthy...
Processing images from: train/Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot...
Processing images from: train/Grape___Black_rot...
Processing images from: train/Tomato___Leaf_Mold...
Processing images from: train/Potato___Late_blight...
Processing images from: train/Tomato___Spider_mites Two-spotted_spider_mite...
Processing images from: train/Tomato___Tomato_mosaic_virus...
Processing images from: train/Cherry_(including_sour)___Powdery_mildew...
Processing images from: train/Grape___Leaf_blight_(Isariopsis_Leaf_Spot)...
Processing images from: train/Tomato___Early_blight...
Processing images from: train/Grape___healthy...
Processing images from: train/Tomato___Late_blight...
Processing images from: train/Apple___Apple_scab...
Processing images from: train/Pepper,_bell___Bacterial_spot...
Processin

## 4.1.Load created csv



In [ ]:
df=pd.read_csv('leaf_features.csv')

In [ ]:
df

,h_mean,h_std,h_skew,s_mean,s_std,s_skew,a_mean,a_std,a_skew,b_mean,b_std,b_skew,contrast,homogeneity,energy,correlation,category_name,label
0,120.424125,48.776633,-0.970756,33.082871,28.647650,1.465646,127.537484,11.415566,-1.210688,131.536844,13.906808,1.318805,191.258374,0.148551,0.022001,0.869564,Tomato___Septoria_leaf_spot,1
1,109.212339,63.717187,-0.462159,73.084287,66.021810,0.786318,127.986573,10.885568,-0.952294,142.525516,23.872952,0.709954,637.799708,0.074567,0.014897,0.772499,Tomato___Septoria_leaf_spot,1
2,90.233100,44.262130,-0.024308,44.480605,31.186917,0.855221,122.256733,11.092356,-0.303879,134.942776,14.421461,0.373949,1370.478290,0.052788,0.009247,0.705274,Tomato___Septoria_leaf_spot,1
3,154.000068,50.888375,-2.400839,30.388264,25.712177,4.280652,133.103591,3.970849,-4.311715,130.160904,8.494991,4.060759,535.876695,0.064746,0.016677,0.669037,Tomato___Septoria_leaf_spot,1
4,116.434892,50.044358,-1.021803,56.575161,51.922295,1.359401,129.332050,11.727681,-1.306548,133.574329,20.562871,1.225886,341.282121,0.146955,0.018911,0.857905,Tomato___Septoria_leaf_spot,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87895,46.822086,19.936092,1.250632,82.847655,82.108411,0.262176,113.993876,14.093161,-0.244340,153.329234,26.094592,0.268971,23.975818,0.360007,0.040127,0.973491,test,1
87896,81.336365,41.277905,0.151765,44.879348,31.766194,1.144183,121.933560,9.414502,-0.157034,135.331890,13.655351,0.238934,942.368968,0.140099,0.027535,0.861334,test,1
87897,39.181380,19.410654,1.702867,68.551544,74.304506,0.622785,116.450493,12.825814,-0.614079,149.378633,23.294092,0.536553,61.992786,0.227352,0.022819,0.969541,test,1
87898,67.805651,41.748435,0.492375,64.328878,35.644535,0.376975,120.649475,10.018828,0.041376,141.568483,18.024250,-0.241005,973.054046,0.096146,0.017361,0.806921,test,1


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# copy the file to drive
!cp leaf_features.csv /content/drive/MyDrive/
print("data set sucsessfully added to drive")

MessageError: Error: credential propagation was unsuccessful